# Aspire : un agent streaming en C# — Channels, BackgroundService, minimal API typée

Ce notebook couvre les grains **A1** et **A2** de l'Epic [#10473](https://github.com/jsboige/CoursIA/issues/10473) — *The Unexpected AI Stack: C#/.NET* (issue [#11516](https://github.com/jsboige/CoursIA/issues/11516)). La ligne de parité visée :

| Situation | Modèle classique (Task/T) | Pattern streaming (.NET) |
|---|---|---|
| Réponse d'un agent | Une seule valeur `Task<string>` à la fin | **`System.Threading.Channels`** : un flux d'événements consommé en temps réel |
| Cycle de vie du service | Boucle manuelle fragile | **`BackgroundService`** : démarrage/arrêt géré par le host |
| Exposition du flux | Endpoint non typé, contrat implicite | **Minimal API typée .NET 10** : requête et résultat fortement typés |

Le fil rouge : un **service d'agent** qui reçoit des demandes en entrée et streame ses événements (tokens, progression, fin) en sortie — le même contrat que les services de la pile GenAI du dépôt, mais exprimé en pur .NET.


## Contexte : pourquoi un flux, pas une valeur ?

Notre pile GenAI (whisper, vLLM, ComfyUI) produit des réponses **par étapes** : un utilisateur qui pose une question à un agent voit les tokens arriver un par un, pas un mur de texte à la fin. Un appel classique `Task<T>` impose d'attendre la réponse complète ; un **flux** (`Channel` + `await foreach`) affiche la progression dès le premier événement.

Le pattern à construire, en trois briques :

1. **Un canal inbound** : les demandes arrivent (channel non borné, faible volume).
2. **Un canal outbound** : le service publie ses événements (tokens, `done`).
3. **Un service hôte** (`BackgroundService`) qui connecte les deux.

Nous commençons par la brique la plus simple : un canal et son flux.


In [1]:
#r "nuget: Microsoft.Extensions.Hosting.Abstractions"

using System.Threading.Channels;
using Microsoft.Extensions.Hosting;

Console.WriteLine($".NET {Environment.Version}");
Console.WriteLine($"Channels : {typeof(Channel<int>).FullName}");
Console.WriteLine($"BackgroundService : {typeof(BackgroundService).FullName}");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Extensions.Hosting.Abstractions, 10.0.11

.NET 10.0.11


Channels : System.Threading.Channels.Channel`1[[System.Int32, System.Private.CoreLib, Version=10.0.0.0, Culture=neutral, PublicKeyToken=7cec85d7bea7798e]]


BackgroundService : Microsoft.Extensions.Hosting.BackgroundService


## A1 — Channels : la file d'événements

`Channel<T>` est une file thread-safe, avec deux faces : un **`Writer`** (le producteur écrit) et un **`Reader`** (le consommateur lit). Le consommateur n'importe pas : dès qu'un élément est écrit, il peut le lire — c'est le cœur du streaming.

Dans l'exemple ci-dessous, l'agent écrit sa réponse **token par token** ; le client lit chaque token dès son arrivée, sans attendre la fin.

In [2]:
// Une réponse d'agent, produite morceau par morceau.
var tokens = new[] { "Bonjour", "je", "suis", "un", "agent", "streaming", "." };
var canal = Channel.CreateUnbounded<string>();

// Producteur : l'agent écrit chaque token des qu'il est disponible.
var producteur = Task.Run(async () =>
{
    foreach (var t in tokens)
    {
        await canal.Writer.WriteAsync(t);
        await Task.Delay(50);
    }
    canal.Writer.Complete();
});

// Consommateur : le client lit le flux en temps reel.
var consommateur = Task.Run(async () =>
{
    await foreach (var t in canal.Reader.ReadAllAsync())
        Console.WriteLine($"recu : {t}");
});

await Task.WhenAll(producteur, consommateur);
Console.WriteLine("Flux termine, canal ferme.");

recu : Bonjour


recu : je


recu : suis


recu : un


recu : agent


recu : streaming


recu : .


Flux termine, canal ferme.


### Interprétation

- **`Writer.WriteAsync`** ne bloque jamais sur un canal non borné : le producteur écrit sans contrainte.
- **`Writer.Complete()`** signale la fin du flux : le consommateur voit sa boucle `await foreach` se terminer.
- **`Reader.ReadAllAsync`** remplace une attente globale par une itération au fil de l'eau.

Sans canal, le producteur devrait bufferiser la réponse entière avant de la rendre. Avec un canal, le premier token est **consommable 50 ms après le début de la production** — c'est la différence entre une réponse monolithique et une réponse en continu.

### Exercice 1 — la backpressure d'un canal borné

Un canal non borné accepte tout ; un canal **borné** (capacité limitée) **bloque le producteur** quand la file est pleine : c'est la **backpressure**. Un consommateur lent ralentit alors le producteur au lieu d'accumuler des millions d'éléments en mémoire. Exécutez d'abord le code tel quel (canal non borné), puis remplacez `CreateUnbounded` par un canal borné de capacité 2 et comparez les temps de blocage du producteur.

In [3]:
// EXERCICE 1 : backpressure avec un canal borne.
// Le producteur emet 5 elements toutes les 10 ms ; le consommateur en traite
// un toutes les 150 ms. Avec un canal NON borne, le producteur ne ralentit
// jamais ; avec un canal borne (capacite 2), il est bloque des que la file
// est pleine.

// TODO Etudiant 1 : remplacer CreateUnbounded par un canal borne :
//     Channel.CreateBounded<int>(new BoundedChannelOptions(2));
// TODO Etudiant 2 : comparer les temps de blocage entre les deux versions
//     et conclure sur la valeur de la backpressure.

var canalEx = Channel.CreateUnbounded<int>();

var producteurEx = Task.Run(async () =>
{
    for (var i = 1; i <= 5; i++)
    {
        var t0 = Environment.TickCount64;
        await canalEx.Writer.WriteAsync(i);
        Console.WriteLine($"[producteur] emet {i} (bloque {Environment.TickCount64 - t0} ms)");
        await Task.Delay(10);
    }
    canalEx.Writer.Complete();
});

var consommateurEx = Task.Run(async () =>
{
    await foreach (var i in canalEx.Reader.ReadAllAsync())
    {
        Console.WriteLine($"[consommateur] traite {i}");
        await Task.Delay(150);
    }
});

await Task.WhenAll(producteurEx, consommateurEx);
Console.WriteLine("Exercice 1 : comparer avec un canal borne de capacite 2.");

[producteur] emet 1 (bloque 0 ms)


[consommateur] traite 1


[producteur] emet 2 (bloque 0 ms)


[producteur] emet 3 (bloque 0 ms)


[producteur] emet 4 (bloque 0 ms)


...

[consommateur] traite 2


[consommateur] traite 3


[consommateur] traite 4


[consommateur] traite 5


Exercice 1 : comparer avec un canal borne de capacite 2.


## A2 — BackgroundService : le service d'agent

Un canal seul ne fait pas un service : il faut un **cycle de vie**. `BackgroundService` fournit la structure : `ExecuteAsync` tourne tant que le host est démarré, et un `CancellationToken` est passé au service pour l'arrêt propre.

Le pattern du service d'agent :

| Canal | Rôle |
|---|---|
| `_inbound` (`Channel<AgentRequest>`) | Les **demandes** entrantes, écrites par le client |
| `_outbound` (`Channel<AgentEvent>`) | Les **événements** publiés par le service, lus par le client |

`ExecuteAsync` consomme `_inbound` et, pour chaque demande, streame les événements sur `_outbound` — exactement le moteur d'un agent de chat.

La cellule suivante déclare uniquement les **types** (record + service) ; ils sont ensuite utilisés dans la cellule d'exécution.

In [4]:
using System.Threading;

// Contrat du service d'agent : des demandes en entree, un flux d'evenements en sortie.
public record AgentRequest(string Prompt);
public record AgentEvent(string Kind, string Payload);

// Le coeur du pattern : un BackgroundService qui consomme le canal inbound
// (les demandes) et streame les evenements sur le canal outbound.
public class StreamingAgentService : BackgroundService
{
    private readonly Channel<AgentRequest> _inbound = Channel.CreateUnbounded<AgentRequest>();
    private readonly Channel<AgentEvent> _outbound = Channel.CreateUnbounded<AgentEvent>();

    public ChannelWriter<AgentRequest> Inbound => _inbound.Writer;
    public ChannelReader<AgentEvent> Outbound => _outbound.Reader;

    protected override async Task ExecuteAsync(CancellationToken stoppingToken)
    {
        try
        {
            await foreach (var req in _inbound.Reader.ReadAllAsync(stoppingToken))
            {
                Console.WriteLine($"[service] recois \"{req.Prompt}\"");
                foreach (var mot in req.Prompt.Split(' '))
                {
                    await _outbound.Writer.WriteAsync(new AgentEvent("token", mot), stoppingToken);
                    await Task.Delay(40, stoppingToken);
                }
                await _outbound.Writer.WriteAsync(new AgentEvent("done", req.Prompt), stoppingToken);
            }
        }
        finally
        {
            _outbound.Writer.Complete();
        }
    }
}

In [5]:
// Demarrage du service sans hote complet : BackgroundService est un IHostedService.
var service = new StreamingAgentService();
var execution = service.StartAsync(CancellationToken.None);

await service.Inbound.WriteAsync(new AgentRequest("Bonjour monde streaming"));
service.Inbound.Complete(); // plus de demandes : le service termine sa boucle.

await foreach (var evt in service.Outbound.ReadAllAsync())
    Console.WriteLine($"[client] {evt.Kind} : {evt.Payload}");

await execution;
Console.WriteLine("Service termine proprement.");

[service] recois "Bonjour monde streaming"


[client] token : Bonjour


[client] token : monde


[client] token : streaming


[client] done : Bonjour monde streaming


Service termine proprement.


### Interprétation

- **`StartAsync`** lance `ExecuteAsync` ; le service tourne en arrière-plan tant qu'on ne l'arrête pas.
- **`Inbound.Complete()`** ferme le canal des demandes : la boucle du service se termine, le `finally` complète `_outbound`, et `await execution` se libère.
- Le client lit le flux **pendant** que le service produit : tokens + `done` arrivent dans l'ordre du traitement.

Dans une application réelle, `StreamingAgentService` est enregistré avec `builder.Services.AddHostedService<...>()` et le host gère le cycle de vie complet (démarrage au boot, arrêt gracieux sur Ctrl+C).

### Exercice 2 — étendre le contrat avec la progression

Le flux actuel émet `token` puis `done`, mais le client ne voit pas **où on en est**. Étendre le contrat pour émettre un événement `progress` avant chaque token (par exemple `ProgressEvent(Produced, Total)`), puis re-exécuter la cellule d'exécution : le client doit afficher la progression.

In [6]:
// EXERCICE 2 : ajouter la progression au flux du service.

Console.WriteLine("Exercice 2 a completer : etendre le contrat du service avec un evenement Progress.");
Console.WriteLine("  1. Declarer un record ProgressEvent(Produced, Total) ci-dessous ;");
Console.WriteLine("  2. Dans StreamingAgentService.ExecuteAsync (cellule des types), ecrire un");
Console.WriteLine("     evenement progress avant chaque token (compter Produced sur Total) ;");
Console.WriteLine("  3. Re-executer la cellule d'execution : le client affiche la progression.");

// TODO Etudiant : declarer ici l'evenement de progression, puis etendre le service.
public record ProgressEvent(int Produced, int Total);

Exercice 2 a completer : etendre le contrat du service avec un evenement Progress.


  1. Declarer un record ProgressEvent(Produced, Total) ci-dessous ;


  2. Dans StreamingAgentService.ExecuteAsync (cellule des types), ecrire un


     evenement progress avant chaque token (compter Produced sur Total) ;


  3. Re-executer la cellule d'execution : le client affiche la progression.


## A3 — Exposer le flux : minimal API typée .NET 10

Le service tourne ; reste à l'**exposer** en HTTP. La minimal API .NET 10 apporte deux briques typées :

- **`TypedResults.Ok(...)` / `TypedResults.Text(...)`** : chaque endpoint retourne un `IResult` **fortement typé** — le compilateur vérifie que l'on ne retourne pas un corps au mauvais type.
- **Handler en classe** : l'endpoint est une classe avec une méthode statique dont les **paramètres sont résolus par le framework** — la requête JSON désérialisée dans un `record`, les services injectés depuis le DI.

Le projet [`StreamingAgent.App/`](StreamingAgent.App/) du dossier réalise le pattern complet : un `AgentService` (BackgroundService + Channels inbound/outbound, comme dans la partie A2) exposé par trois endpoints typés. Le notebook le **lance réellement** (`dotnet run`), l'interroge, puis l'arrête.

> Prérequis : le projet doit être compilé une fois (`dotnet build` dans `StreamingAgent.App/`) — la cellule ci-dessous le fait si besoin via `dotnet run` (build incrémental).

In [7]:
using System.Diagnostics;
using System.IO;
using System.Net.Http;
using System.Text;

// Lance le service d'agent en arriere-plan, requete ses endpoints types, puis l'arrete.
var projet = Path.Combine(Environment.CurrentDirectory, "StreamingAgent.App");
Process proc = null;
try
{
    if (!Directory.Exists(projet))
    {
        Console.WriteLine($"Projet introuvable : {projet}");
        Console.WriteLine("Re-executer ce notebook depuis le dossier MyIA.AI.Notebooks/GenAI/Aspire.");
    }
    else
    {
        proc = Process.Start(new ProcessStartInfo(
            "dotnet", "run --no-build --project \"" + projet + "\" -- --urls http://127.0.0.1:5128")
        {
            WorkingDirectory = projet,
        });

        using var client = new HttpClient { BaseAddress = new Uri("http://127.0.0.1:5128") };

        // Poll du endpoint /health jusqu'a ce que le serveur ecoute.
        HttpResponseMessage health = null;
        for (var i = 0; i < 40 && health is null; i++)
        {
            try { health = await client.GetAsync("/health"); }
            catch (HttpRequestException) { await Task.Delay(250); }
        }
        var healthBody = health is null ? "indisponible" : await health.Content.ReadAsStringAsync();
        Console.WriteLine($"/health -> {(int)(health?.StatusCode ?? 0)} : {healthBody}");

        var corpsGreet = new StringContent("{\"name\":\"Claude\"}", Encoding.UTF8, "application/json");
        var greet = await client.PostAsync("/greet", corpsGreet);
        Console.WriteLine($"/greet -> {(int)greet.StatusCode} : {await greet.Content.ReadAsStringAsync()}");

        var corpsFlux = new StringContent("{\"prompt\":\"Bonjour monde streaming\"}", Encoding.UTF8, "application/json");
        var flux = await client.PostAsync("/stream", corpsFlux);
        Console.WriteLine($"/stream -> {(int)flux.StatusCode} : {await flux.Content.ReadAsStringAsync()}");
    }
}
finally
{
    if (proc is not null)
    {
        try { proc.Kill(entireProcessTree: true); } catch (InvalidOperationException) { }
        await proc.WaitForExitAsync();
        Console.WriteLine("Service arrete (processus termine).");
    }
}

/health -> 200 : {"status":"ok","service":"streaming-agent"}


/greet -> 200 : {"message":"Bonjour Claude depuis un endpoint typé .NET 10."}


/stream -> 200 : [{"kind":"token","payload":"Bonjour"},{"kind":"token","payload":"monde"},{"kind":"token","payload":"streaming"},{"kind":"done","payload":"Bonjour monde streaming"}]


Service arrete (processus termine).


### Interprétation

- **`app.MapGet("/health", () => TypedResults.Ok(new HealthResponse(...)))`** : le résultat est un `Ok<HealthResponse>` — le type de la réponse est vérifié à la compilation.
- **`GreetHandler.Handle`** : la classe d'endpoint reçoit la requête JSON **désérialisée dans le `record GreetRequest`** — plus de `FromBody` implicite ni de dictionnaire non typé.
- **`/stream`** : l'endpoint écrit la demande dans le canal **inbound** du service, lit le flux **outbound** jusqu'à l'événement `done`, et retourne le JSON typé. Les tokens sont produits toutes les 80 ms par `AgentService` — la réponse montre la séquence réelle.

Voir [`StreamingAgent.App/Program.cs`](StreamingAgent.App/Program.cs) pour l'implémentation complète du service et des endpoints.

### Exercice 3 — concevoir l'endpoint typé du service

La minimal API .NET 10 expose des endpoints « en classe » : une **requête fortement typée** en entrée, un **résultat fortement typé** en sortie. Modéliser le même contrat, puis l'appliquer au service du notebook.

In [8]:
using System.Threading;

// EXERCICE 3 : concevoir l'endpoint type du service.

Console.WriteLine("Exercice 3 a completer : implementer l'endpoint type du service.");
Console.WriteLine("  1. Declarer GreetRequest / GreetResponse (records) ci-dessous ;");
Console.WriteLine("  2. Declarer l'interface IStreamEndpoint<in T> avec HandleAsync ;");
Console.WriteLine("  3. Implementer GreetEndpoint : IStreamEndpoint<GreetRequest> retournant");
Console.WriteLine("     la reponse typee (voir StreamingAgent.App/Program.cs pour le modele reel).");

// TODO Etudiant : completer le contrat et l'implementation.
public record GreetRequest(string Name);
public record GreetResponse(string Message);

public interface IStreamEndpoint<in T>
{
    Task<GreetResponse> HandleAsync(T request, CancellationToken ct);
}

Exercice 3 a completer : implementer l'endpoint type du service.


  1. Declarer GreetRequest / GreetResponse (records) ci-dessous ;


  2. Declarer l'interface IStreamEndpoint<in T> avec HandleAsync ;


  3. Implementer GreetEndpoint : IStreamEndpoint<GreetRequest> retournant


     la reponse typee (voir StreamingAgent.App/Program.cs pour le modele reel).


## Conclusion

Le pattern complet d'un agent streaming en C# tient en trois briques qui se composent :

| Brique | API | Rôle |
|---|---|---|
| File d'événements | `System.Threading.Channels` | `Writer`/`Reader`, backpressure, flux `await foreach` |
| Cycle de vie | `BackgroundService` | `ExecuteAsync`, arrêt coopératif par `CancellationToken` |
| Exposition | Minimal API typée .NET 10 | `TypedResults`, handler en classe, injection DI |

Dans le projet [`StreamingAgent.App/`](StreamingAgent.App/), ces trois briques forment un service réel : un `AgentService` hôte (canaux inbound/outbound) exposé par des endpoints typés (`/health`, `/greet`, `/stream`), vérifiable par `curl`. Le même contrat — flux d'événements, cycle de vie hôte, endpoints typés — gouverne les services de la pile GenAI orchestrée par Aspire (les notebooks [01](01-Aspire-Orchestration-GenAi.ipynb) et [02](02-Aspire-GenAiStack-Reel.ipynb)).

**Pour aller plus loin** : brancher l'observabilité (même famille, grain [#11516](https://github.com/jsboige/CoursIA/issues/11516) A9) — `ActivitySource` au moment de chaque token, trace OTLP du flux complet. Le mécanisme de l'[observabilité dans la série SemanticKernel](../SemanticKernel/04-Filters-Observability.ipynb) s'applique tel quel à ce service.
